# Independent dataset: NF-ToN-IoT-v2 (R1-3)
Runtime > Run all. Allow Google Drive access when asked. The data file is read from ColabNotebooks/NF-ToN-IoT-v2/NF-ToN-IoT-V2.parquet. The notebook prints the categories it finds and the features it builds; if a column name differs from the expected one it stops and prints the real column names. It then trains the IDS and detector on clean flows, attacks original malware flows with our black-box, adaptive and gradient attackers, and reports how the ensemble detector (and two comparison detectors) behave.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
except ImportError:
    drive = None                      # not running in Colab
if drive is not None:
    drive.mount("/content/drive")     # errors are shown, not hidden
    assert os.path.exists("/content/drive/MyDrive"), "Google Drive did not mount - re-run and click Allow"

# ---------------- CONFIG (edit here) ----------------
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
FAMILIES = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "CICIDS2018": ["Zeus_Ares"]}
ATTACKS = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]
SEEDS = list(range(int(os.environ.get("N_SEEDS", "5"))))
N_CHUNKS = int(os.environ.get("N_CHUNKS", "20"))       # position blocks
BUFFER = 0.02                                          # drop rows within 2% of a chunk edge
MAX_BENIGN = int(os.environ.get("MAX_BENIGN", "150000"))        # systematic (ordered) subsample of benign
MAX_TRAIN_PER_CLASS = int(os.environ.get("MAX_TRAIN", "40000"))
MAX_EVAL_PER_SET = int(os.environ.get("MAX_EVAL", "30000"))     # cap per val/test set (speed)
FAR_TARGET_PCT = 95                                    # threshold = 95th percentile of clean validation scores
BITS = 8                                               # quantization bits for the cleaned-up squeezer
DROP = {"Unnamed: 0", "Label", "label", "n_actions"}
os.makedirs(OUT, exist_ok=True)


In [ ]:
# ---------------- DATA ----------------
def with_pos(df):
    df = df.copy()
    df["_pos"] = (np.arange(len(df)) + 0.5) / len(df)
    return df

def load_dataset(ds):
    d = f"{BASE}/{ds}"
    benign = pd.read_csv(f"{d}/benign.csv")
    if len(benign) > MAX_BENIGN:                       # ordered subsample, keeps temporal structure
        benign = benign.iloc[np.linspace(0, len(benign) - 1, MAX_BENIGN).astype(int)]
    fams = FAMILIES[ds]
    orig = [with_pos(pd.read_csv(f"{d}/malicious/{f}.csv")) for f in fams]
    adv = {(o, a): [with_pos(pd.read_csv(f"{d}/{o}/{a}/{f}.csv")) for f in fams] for o, a in ATTACKS}
    benign = with_pos(benign)
    ref = [benign] + orig + [x for v in adv.values() for x in v]
    feats = [c for c in benign.columns
             if c not in DROP and c != "_pos"
             and all(c in r.columns for r in ref)
             and all(pd.api.types.is_numeric_dtype(r[c]) for r in ref)]
    def pool(frames):
        df = pd.concat(frames, ignore_index=True)
        X = df[feats].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).values.astype(np.float32)
        return X, df["_pos"].values
    data = {"benign": pool([benign]), "orig": pool(orig)}
    for k, v in adv.items():
        data[k] = pool(v)
    return data, feats

def split_roles(seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N_CHUNKS)
    n_tr, n_va = int(N_CHUNKS * 0.5), int(N_CHUNKS * 0.2)
    role = np.empty(N_CHUNKS, dtype=object)
    role[perm[:n_tr]] = "train"
    role[perm[n_tr:n_tr + n_va]] = "val"
    role[perm[n_tr + n_va:]] = "test"
    return role

def take(pool, role, split, rng, cap):
    X, pos = pool
    ch = np.minimum((pos * N_CHUNKS).astype(int), N_CHUNKS - 1)
    frac = pos * N_CHUNKS - ch
    m = (role[ch] == split) & (frac > BUFFER) & (frac < 1 - BUFFER)
    Xs = X[m]
    if len(Xs) > cap:
        Xs = Xs[rng.choice(len(Xs), cap, replace=False)]
    return Xs


In [ ]:
# ---------------- SQUEEZERS ----------------
def make_quant(Xtr, bits=BITS, q=(0.5, 99.5)):
    """True k-bit quantization: clip to the training [q_lo, q_hi] range, then 2^k levels per feature."""
    lo = np.percentile(Xtr, q[0], axis=0); hi = np.percentile(Xtr, q[1], axis=0)
    span = np.maximum(hi - lo, 1e-8); L = 2 ** bits - 1
    def sq(X):
        Z = np.clip((X - lo) / span, 0, 1)
        return (np.round(Z * L) / L * span + lo).astype(np.float32)
    return sq

def paper_sq(X):
    """The original paper code: round(x*256)/256, then a trailing median over 3 adjacent feature columns."""
    X1 = np.round(X * 256) / 256
    s1 = np.full_like(X1, np.nan); s1[:, 1:] = X1[:, :-1]
    s2 = np.full_like(X1, np.nan); s2[:, 2:] = X1[:, :-2]
    return np.nanmedian(np.stack([X1, s1, s2]), axis=0).astype(np.float32)

# ---------------- SCORES ----------------
def conf(p): return np.maximum(p, 1 - p)

def base_probs(X, rf, xgb):
    return rf.predict_proba(X)[:, 1], xgb.predict_proba(X)[:, 1]

def independent_scores(X, rf, xgb, rng):
    """Detectors that do not depend on the squeezer."""
    p_rf, p_xg = base_probs(X, rf, xgb)
    pn = rf.predict_proba(X + rng.normal(0, 0.01, X.shape).astype(np.float32))[:, 1]
    return {"Noise sensitivity (RF)": np.abs(p_rf - pn),
            "Ensemble disagreement only": np.abs(p_rf - p_xg)}

def squeeze_scores(X, rf, xgb, sq):
    p_rf, p_xg = base_probs(X, rf, xgb)
    Xs = sq(X)
    s_rf, s_xg = base_probs(Xs, rf, xgb)
    return {"Feature squeezing (RF)": np.abs(p_rf - s_rf),
            "Confidence drop (RF)": np.abs(conf(p_rf) - conf(s_rf)),
            "Squeeze + ensemble, Eq.11 variance": np.var(np.vstack([s_rf, s_xg]), axis=0),
            "Squeeze + ensemble, Eq.24 sum": np.abs(p_rf - s_rf) + np.abs(p_xg - s_xg)}

PROPOSED = ["Squeeze + ensemble, Eq.11 variance", "Squeeze + ensemble, Eq.24 sum"]
BASELINES = ["Noise sensitivity (RF)", "Feature squeezing (RF)", "Confidence drop (RF)"]

# ---------------- METRICS ----------------
def metrics(neg_val, neg_te, pos_te):
    thr = np.percentile(neg_val, FAR_TARGET_PCT)
    tp = int((pos_te > thr).sum()); fn = len(pos_te) - tp
    fp = int((neg_te > thr).sum()); tn = len(neg_te) - fp
    tpr, fpr = tp / len(pos_te), fp / len(neg_te)
    prec = tpr / (tpr + fpr + 1e-12)                  # precision at balanced prevalence
    f1 = 2 * prec * tpr / (prec + tpr + 1e-12)
    y = np.r_[np.zeros(len(neg_te)), np.ones(len(pos_te))]; s = np.r_[neg_te, pos_te]
    return dict(thr=thr, TPR=tpr, FPR=fpr, Precision_bal=prec, F1_bal=f1,
                AUROC=roc_auc_score(y, s), AUPRC=average_precision_score(y, s),
                TP=tp, FN=fn, FP=fp, TN=tn)


In [ ]:
# ---- INDEPENDENT DATASET (R1-3): NF-ToN-IoT-v2 with our own attackers (no DReLAB data) ----
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import roc_auc_score
NF_PATH = os.environ.get("NF_PATH", "/content/drive/MyDrive/ColabNotebooks/NF-ToN-IoT-v2/NF-ToN-IoT-V2.parquet")
NF_BENIGN_CAP, NF_CAT_CAP, NF_MIN_CAT = 200000, 40000, 5000   # benign flows kept, flows kept per attack category, minimum size of a category used
COL = dict(in_bytes="IN_BYTES", out_bytes="OUT_BYTES", in_pkts="IN_PKTS", out_pkts="OUT_PKTS", dur_ms="FLOW_DURATION_MILLISECONDS",
           proto="PROTOCOL", l7="L7_PROTO", flags="TCP_FLAGS", dport="L4_DST_PORT", attack="Attack")
N_ATTACK, ITERS, SIGMA, FMAX = 2000, 60, 0.5, 10.0
WB_STEPS, WB_ALPHA, WB_EPS, SURR_CAP, AE_CAP, AE_EPOCHS = 30, 0.15, 0.05, 30000, 30000, 40
NEED = ["Dur", "InBytes", "OutBytes", "TotPkts", "TotBytes", "BytesPerPkt", "BytesPerSec", "PktsPerSec", "RatioOutIn"]

def nf_batches(columns, batch=500_000):
    if NF_PATH.endswith(".parquet"):
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(NF_PATH)
        for b in pf.iter_batches(batch_size=batch, columns=columns): yield b.to_pandas()
    else:
        for ch in pd.read_csv(NF_PATH, usecols=columns, chunksize=batch): yield ch

def nf_columns():
    if NF_PATH.endswith(".parquet"):
        import pyarrow.parquet as pq; return pq.ParquetFile(NF_PATH).schema_arrow.names
    return list(pd.read_csv(NF_PATH, nrows=1).columns)

def nf_features(df):
    f = lambda c: df[COL[c]].astype(float).values
    inb, outb, ip, op, dur = f("in_bytes"), f("out_bytes"), f("in_pkts"), f("out_pkts"), f("dur_ms") / 1000.0
    pk, tot, dp = ip + op, inb + outb, f("dport")
    cols = {"Dur": dur, "InBytes": inb, "OutBytes": outb, "TotPkts": pk, "TotBytes": tot,
            "BytesPerPkt": tot / np.maximum(pk, 1), "BytesPerSec": tot / np.maximum(dur, 1e-3), "PktsPerSec": pk / np.maximum(dur, 1e-3),
            "RatioOutIn": outb / np.maximum(inb, 1), "PROTOCOL": f("proto"), "L7_PROTO": f("l7"), "TCP_FLAGS": f("flags"),
            "DstPort_Well_Known": (dp < 1024).astype(float), "DstPort_Registered": ((dp >= 1024) & (dp < 49152)).astype(float), "DstPort_Private": (dp >= 49152).astype(float)}
    return pd.DataFrame(cols)

def load_nf():
    names = nf_columns(); missing = [v for v in COL.values() if v not in names]
    if missing:
        print("Columns in the file:", names); raise ValueError(f"Expected columns not found: {missing}. Edit the COL mapping at the top of this cell.")
    use = list(COL.values()); acol = COL["attack"]
    counts = {}
    for b in nf_batches([acol]):
        for k, v in b[acol].astype(str).value_counts().items(): counts[k] = counts.get(k, 0) + int(v)
    print("Flows per category:", counts)
    ben = [k for k in counts if k.lower() == "benign"]; mal = [k for k, v in counts.items() if k.lower() != "benign" and v >= NF_MIN_CAT]
    print("Benign category:", ben, "| malware categories used:", mal, "| skipped (too small):", [k for k in counts if k not in ben + mal])
    keep_p = {k: min(1.0, (NF_BENIGN_CAP if k in ben else NF_CAT_CAP) / counts[k]) for k in ben + mal}
    rng = np.random.RandomState(0); parts = {k: [] for k in keep_p}
    for b in nf_batches(use):
        b = b[b[acol].astype(str).isin(keep_p)]
        keep = rng.random_sample(len(b)) < b[acol].astype(str).map(keep_p).values
        b = b[keep]
        for k, g in b.groupby(b[acol].astype(str), sort=False): parts[k].append(g)
    def pool(keys):
        dfs = []
        for k in keys:
            g = pd.concat(parts[k], ignore_index=True); F = nf_features(g); F["_pos"] = (np.arange(len(F)) + 0.5) / len(F); dfs.append(F)
        F = pd.concat(dfs, ignore_index=True); pos = F.pop("_pos").values
        X = F.replace([np.inf, -np.inf], np.nan); feats = list(X.columns); X = X.fillna(X.median()).values.astype(np.float32); return X, pos, feats
    Xb, pb, feats = pool(ben); Xo, po, _ = pool(mal)
    print(f"benign flows: {len(Xb)}, original malware flows: {len(Xo)}, features ({len(feats)}): {feats}")
    return {"benign": (Xb, pb), "orig": (Xo, po)}, feats

def split_sets(D, seed):
    role = split_roles(seed); rng = np.random.RandomState(seed)
    g = lambda k, sp, cap: take(D[k], role, sp, rng, cap)
    Xb, Xo = g("benign", "train", 10**9), g("orig", "train", 10**9)
    n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
    Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
    nb_, no_ = g("benign", "test", MAX_EVAL_PER_SET), g("orig", "test", MAX_EVAL_PER_SET)
    return dict(train=(np.vstack([Xb, Xo]), np.r_[np.zeros(n), np.ones(n)]),
                val=np.vstack([g("benign", "val", MAX_EVAL_PER_SET), g("orig", "val", MAX_EVAL_PER_SET)]),
                neg=np.vstack([nb_, no_]), neg_b=nb_, neg_o=no_)

def perturb(X0, F, ix, rnd=True):
    X = X0.copy(); r = np.round if rnd else (lambda a: a)
    dur0, in0, out0, pk0, tot0 = (X0[:, ix[c]] for c in ("Dur", "InBytes", "OutBytes", "TotPkts", "TotBytes"))
    dur = dur0 * F[:, 0]; inb = r(in0 * F[:, 1]); outb = r(out0 * F[:, 2]); pk = np.maximum(r(pk0 * F[:, 3]), 1)
    tot = tot0 + (inb - in0) + (outb - out0)
    ratio = lambda a, b: np.where(b > 0, a / np.maximum(b, 1e-12), 1.0)
    kd, kt, kp, ko, ki = ratio(dur, dur0), ratio(tot, tot0), ratio(pk, pk0), ratio(outb, out0), ratio(inb, in0)
    X[:, ix["Dur"]] = dur; X[:, ix["InBytes"]] = inb; X[:, ix["OutBytes"]] = outb; X[:, ix["TotPkts"]] = pk; X[:, ix["TotBytes"]] = tot
    X[:, ix["BytesPerPkt"]] = X0[:, ix["BytesPerPkt"]] * kt / np.maximum(kp, 1e-12)
    X[:, ix["BytesPerSec"]] = X0[:, ix["BytesPerSec"]] * kt / np.maximum(kd, 1e-12)
    X[:, ix["PktsPerSec"]] = X0[:, ix["PktsPerSec"]] * kp / np.maximum(kd, 1e-12)
    X[:, ix["RatioOutIn"]] = X0[:, ix["RatioOutIn"]] * ko / np.maximum(ki, 1e-12)
    return X

def run_attack(kind, X0, ix, T, rf, xg, tau, rng):
    n = len(X0); F = np.ones((n, 4))
    obj = {"A": lambda pr, px: pr, "B": lambda pr, px: np.maximum(pr, px),
           "C": lambda pr, px: np.maximum(pr, px) + 5 * np.maximum(0, np.abs(pr - px) - tau)}[kind]
    Z0 = T(X0); pr, px = rf.predict_proba(Z0)[:, 1], xg.predict_proba(Z0)[:, 1]; cur = obj(pr, px)
    for it in range(ITERS):
        Fc = np.clip(F * np.exp(rng.normal(0, SIGMA, F.shape)), 1, FMAX)
        Zn = T(perturb(X0, Fc, ix)); prn, pxn = rf.predict_proba(Zn)[:, 1], xg.predict_proba(Zn)[:, 1]
        new = obj(prn, pxn); b = new < cur
        F[b] = Fc[b]; pr[b] = prn[b]; px[b] = pxn[b]; cur[b] = new[b]
    return perturb(X0, F, ix)

def mlp_logit_grad(mlp, Z):
    a, pres = Z, []
    for W, b in zip(mlp.coefs_[:-1], mlp.intercepts_[:-1]):
        pre = a @ W + b; pres.append(pre); a = np.maximum(pre, 0)
    g = np.repeat(mlp.coefs_[-1].T, len(Z), axis=0)
    for W, pre in zip(reversed(mlp.coefs_[:-1]), reversed(pres)): g = (g * (pre > 0)) @ W.T
    return g

def whitebox_pgd(X0, ix, scaler, mlp):
    X0 = X0.astype(np.float64); n = len(X0); L = np.zeros((n, 4)); LMAX = np.log(FMAX)
    for _ in range(WB_STEPS):
        X = perturb(X0, np.exp(L), ix, rnd=False)
        gX = mlp_logit_grad(mlp, ((X - scaler.mean_) / scaler.scale_).astype(np.float32)) / scaler.scale_
        grad = np.zeros((n, 4))
        for k in range(4):
            Lk = L.copy(); Lk[:, k] += WB_EPS
            grad[:, k] = ((perturb(X0, np.exp(Lk), ix, rnd=False) - X) * gX).sum(1) / WB_EPS
        L = np.clip(L - WB_ALPHA * np.sign(grad), 0, LMAX)
    return perturb(X0, np.exp(L), ix, rnd=True)

D, feats = load_nf(); ix = {c: feats.index(c) for c in NEED}
att_rows, det_rows, ids_rows = [], [], []
ATT = {"A: black-box vs RF only": "A", "B: black-box vs RF and XGBoost": "B", "C: adaptive (knows detector and threshold)": "C", "G: gradient attack on a surrogate (transfer)": "G"}
for seed in SEEDS:
    t0 = time.time()
    S = split_sets(D, seed); Xtr, ytr = S["train"]
    sc = StandardScaler().fit(Xtr); T = lambda X, sc=sc: sc.transform(X).astype(np.float32); Z = T(Xtr)
    val, neg = T(S["val"]), T(S["neg"])
    rf = RandomForestClassifier(150, max_depth=20, n_jobs=-1, random_state=seed).fit(Z, ytr)
    xg = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss", random_state=seed).fit(Z, ytr)
    ens = lambda Zx: np.abs(rf.predict_proba(Zx)[:, 1] - xg.predict_proba(Zx)[:, 1])
    tau = np.percentile(ens(val), FAR_TARGET_PCT); s_neg = ens(neg)
    idx = np.random.RandomState(seed).choice(len(Z), min(len(Z), AE_CAP), replace=False)
    ae = MLPRegressor(hidden_layer_sizes=(64, 16, 64), activation="relu", solver="adam", batch_size=256, max_iter=AE_EPOCHS,
                      early_stopping=False, tol=0.0, n_iter_no_change=AE_EPOCHS + 1, random_state=seed).fit(Z[idx], Z[idx])
    sq = make_quant(Z)
    mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", early_stopping=True, max_iter=60, random_state=seed).fit(Z[idx], ytr[idx])
    ids_rows.append(dict(seed=seed, clean_acc=float(np.r_[rf.predict(T(S["neg_b"])) == 0, rf.predict(T(S["neg_o"])) == 1].mean())))
    X_all = take(D["orig"], split_roles(seed), "test", np.random.RandomState(1000 + seed), 5 * N_ATTACK)
    X0 = X_all[rf.predict_proba(T(X_all))[:, 1] >= 0.5][:N_ATTACK]
    adv_sets = {}
    for name, kind in ATT.items():
        Xa = whitebox_pgd(X0, ix, sc, mlp).astype(np.float32) if kind == "G" else run_attack(kind, X0, ix, T, rf, xg, tau, np.random.RandomState(seed)).astype(np.float32)
        Za = T(Xa); pr, px = rf.predict_proba(Za)[:, 1], xg.predict_proba(Za)[:, 1]; Dd = np.abs(pr - px)
        ev_rf = pr < 0.5; ev_both = ev_rf & (px < 0.5); fl = Dd > tau; adv_sets[name] = (Za, ev_rf)
        att_rows.append(dict(seed=seed, attack=name, n=len(X0), evade_RF=ev_rf.mean(), evade_both=ev_both.mean(), flagged_all=fl.mean(),
                             flagged_among_RF_evaders=(fl[ev_rf].mean() if ev_rf.any() else np.nan), undetected_RF_evasion=(ev_rf & ~fl).mean(),
                             AUROC=roc_auc_score(np.r_[np.zeros(len(s_neg)), np.ones(len(Dd))], np.r_[s_neg, Dd])))
    pos_all = np.vstack([v[0] for v in adv_sets.values()]); pos_ev = np.vstack([v[0][v[1]] for v in adv_sets.values()])
    dets = {"Ensemble disagreement": ens,
            "Squeeze + ensemble, Eq.11 (8-bit)": lambda X: squeeze_scores(X, rf, xg, sq)["Squeeze + ensemble, Eq.11 variance"],
            "Autoencoder (fixed 40 epochs)": lambda X: ((ae.predict(X) - X) ** 2).mean(axis=1)}
    for dn, f in dets.items():
        sv, sn = f(val), f(neg)
        for subset, P in (("all attacked flows", pos_all), ("IDS-evading flows only", pos_ev)):
            if len(P) >= 100:
                m = metrics(sv, sn, f(P)); det_rows.append(dict(seed=seed, detector=dn, subset=subset, n_pos=len(P), TPR=m["TPR"], FPR=m["FPR"], AUROC=m["AUROC"], AUPRC=m["AUPRC"]))
    print(f"  seed {seed} done in {time.time() - t0:.0f}s ({len(X0)} flows attacked)")

att = pd.DataFrame(att_rows); det = pd.DataFrame(det_rows); ids = pd.DataFrame(ids_rows)
att.to_csv(f"{OUT}/nf_attack_results.csv", index=False); det.to_csv(f"{OUT}/nf_detector_results.csv", index=False); ids.to_csv(f"{OUT}/nf_ids_sanity.csv", index=False)
pm = lambda x: f"{np.mean(x):.3f}+/-{np.std(x, ddof=1) if len(x) > 1 else 0:.3f}"
pd.set_option("display.width", 250, "display.max_colwidth", 50)
print("\nIDS accuracy on clean test flows:", pm(ids.clean_acc))
print("\n===== NF-ToN-IoT-v2: attackers against IDS + ensemble detector (mean +/- std over seeds) =====")
print(att.groupby("attack")[["evade_RF", "evade_both", "flagged_all", "flagged_among_RF_evaders", "undetected_RF_evasion", "AUROC"]].agg(pm).reset_index().to_string(index=False))
print("\n===== NF-ToN-IoT-v2: detector comparison on the pooled attacked flows (threshold = 95th percentile of clean validation scores) =====")
print(det.groupby(["subset", "detector"])[["TPR", "FPR", "AUROC", "AUPRC"]].agg(pm).reset_index().to_string(index=False))
